# JachaiX — Knowledge Base Crawler
Scrapes 4 trusted Bangladeshi news/fact-checking sources and saves articles as JSON to `corpus/raw/`.

**Sources:**
1. Rumor Scanner BD — dedicated BD fact-checker (highest reliability: 0.95)
2. BBC Bangla — international authority (0.90)
3. Prothom Alo — largest Bangla newspaper (0.80)
4. Daily Star BD — top English BD paper (0.80)

**Output:** `e:/jachaix/corpus/raw/<source>_<timestamp>.json`

Run each cell one at a time. If one source fails, fix that cell only — others are unaffected.

In [16]:
## Cell 1 — Imports & Setup
import os, sys, json, time, hashlib
from datetime import datetime
from pathlib import Path
import requests
from bs4 import BeautifulSoup

# Output directory
RAW_DIR = Path("E:/jachaix/corpus/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)

# Common headers to avoid bot detection
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/120.0 Safari/537.36",
    "Accept-Language": "bn-BD,bn;q=0.9,en-US;q=0.8,en;q=0.7",
}

# Source reliability scores
RELIABILITY = {
    "rumorscanner": 0.95,
    "bbc_bangla":   0.90,
    "prothom_alo":  0.80,
    "daily_star":   0.80,
}

def save_article(article: dict, source: str):
    """Save a single article as JSON. Skip if URL already saved."""
    url_hash = hashlib.md5(article["url"].encode()).hexdigest()[:10]
    filepath = RAW_DIR / f"{source}_{url_hash}.json"
    if filepath.exists():
        return False  # already saved
    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(article, f, ensure_ascii=False, indent=2)
    return True

def count_saved():
    return len(list(RAW_DIR.glob("*.json")))

print(f"Output dir: {RAW_DIR}")
print(f"Already saved: {count_saved()} articles")
print("Libraries OK — ready to crawl!")

Output dir: E:\jachaix\corpus\raw
Already saved: 189 articles
Libraries OK — ready to crawl!


## Source 1 — Rumor Scanner BD
**URL:** https://www.rumorscanner.com  
**Why:** Only dedicated Bangla fact-checking site in Bangladesh. Every article has a verdict (true/false/misleading).  
**Reliability:** 0.95  
**Strategy:** Scrape category listing pages → follow article links → extract title + body text.

In [2]:
## Source 1 — Rumor Scanner BD crawler

SOURCE = "rumorscanner"
BASE_URL = "https://www.rumorscanner.com"
CATEGORY_PAGES = [
    "https://www.rumorscanner.com/category/fact-check/",
    "https://www.rumorscanner.com/category/fact-check/page/2/",
    "https://www.rumorscanner.com/category/fact-check/page/3/",
    "https://www.rumorscanner.com/category/fact-check/page/4/",
    "https://www.rumorscanner.com/category/fact-check/page/5/",
]

def get_rumorscanner_links(page_url):
    """Get article links from a category listing page."""
    try:
        r = requests.get(page_url, headers=HEADERS, timeout=15)
        r.raise_for_status()
        soup = BeautifulSoup(r.text, "lxml")
        links = []
        for a in soup.select("h2.entry-title a, h3.entry-title a, .post-title a"):
            href = a.get("href", "")
            if href.startswith("http") and "rumorscanner.com" in href:
                links.append(href)
        return list(set(links))
    except Exception as e:
        print(f"  [WARN] Failed to get links from {page_url}: {e}")
        return []

def scrape_rumorscanner_article(url):
    """Scrape a single Rumor Scanner article."""
    try:
        r = requests.get(url, headers=HEADERS, timeout=15)
        r.raise_for_status()
        soup = BeautifulSoup(r.text, "lxml")

        title = ""
        for sel in ["h1.entry-title", "h1.post-title", "h1"]:
            t = soup.select_one(sel)
            if t:
                title = t.get_text(strip=True)
                break

        content = ""
        body = soup.select_one("div.entry-content, div.post-content, article")
        if body:
            paragraphs = body.find_all("p")
            content = " ".join(p.get_text(strip=True) for p in paragraphs if len(p.get_text(strip=True)) > 30)

        date = ""
        d = soup.select_one("time[datetime], .entry-date, .post-date")
        if d:
            date = d.get("datetime", d.get_text(strip=True))

        if not title or len(content) < 100:
            return None

        return {
            "source": SOURCE,
            "url": url,
            "title": title,
            "content": content,
            "language": "bn",
            "published_date": date,
            "reliability_score": RELIABILITY[SOURCE],
            "scraped_at": datetime.utcnow().isoformat(),
        }
    except Exception as e:
        print(f"  [WARN] Failed to scrape {url}: {e}")
        return None

# Run crawler
print("Collecting article links from Rumor Scanner BD...")
all_links = []
for page in CATEGORY_PAGES:
    links = get_rumorscanner_links(page)
    print(f"  {page} → {len(links)} links")
    all_links.extend(links)
    time.sleep(1)

all_links = list(set(all_links))
print(f"\nTotal unique links: {len(all_links)}")
print("Scraping articles...")

saved = 0
failed = 0
for i, url in enumerate(all_links[:50]):  # cap at 50 articles per source
    article = scrape_rumorscanner_article(url)
    if article:
        if save_article(article, SOURCE):
            saved += 1
            print(f"  [{i+1}] SAVED: {article['title'][:60]}")
        else:
            print(f"  [{i+1}] SKIP (duplicate): {url}")
    else:
        failed += 1
    time.sleep(1.5)  # polite delay

print(f"\nRumor Scanner BD — Saved: {saved} | Failed: {failed}")
print(f"Total in corpus: {count_saved()} articles")

  https://www.rumorscanner.com/category/fact-check/ → 12 links
  https://www.rumorscanner.com/category/fact-check/page/2/ → 12 links
  https://www.rumorscanner.com/category/fact-check/page/3/ → 12 links
  https://www.rumorscanner.com/category/fact-check/page/4/ → 12 links
  https://www.rumorscanner.com/category/fact-check/page/5/ → 12 links

Total unique links: 48
Scraping articles...
  [1] SAVED: ‘নুরুর মতো পাটোয়ারীকে মারতে মারতে মন্ত্রী বানাবে বিএনপি’ খা
  [2] SAVED: খুলনা আদালত প্রাঙ্গণে শিশু ধর্ষণকারীকে গুলি করে হত্যার দাবিট
  [3] SAVED: হাম ইস্যুতে মানজুর আল মতিনের নামে ভুয়া মন্তব্য প্রচার
  [4] SAVED: ভারতের সংসদে শুভেন্দু অধিকারীর ওপর জুতা নিক্ষেপের দৃশ্য দাবি
  [5] SAVED: প্রধানমন্ত্রী কর্তৃক রামিসার বাবাকে পাঁচ লাখ টাকা দেওয়ার ভু
  [6] SAVED: “মারা গেলে দিল্লির শ্মশানে সৎকার”—শেখ হাসিনার নামে ভুয়া মন্
  [7] SAVED: রামিসা হত্যাকাণ্ড নিয়ে অভিনেতা দেবের নামে ভুয়া মন্তব্য প্র
  [8] SAVED: রামিসা হত্যায় অভিযুক্ত সোহেলের সঙ্গে প্রতিমন্ত্রী আমিনুল হক
  [9] SAVED: ময়মনসিংহে শি

## Source 2 — BBC Bangla
**URL:** https://www.bbc.com/bengali  
**Why:** International authority, high credibility, Bangla language coverage of BD events.  
**Reliability:** 0.90  
**Strategy:** RSS feed (most reliable, no blocking) → follow article links → extract full text.

In [3]:
## Source 2 — BBC Bangla crawler (via RSS)
import xml.etree.ElementTree as ET

SOURCE = "bbc_bangla"
RSS_FEEDS = [
    "https://feeds.bbci.co.uk/bengali/rss.xml",
]

def get_bbc_links_from_rss(rss_url):
    """Parse BBC Bangla RSS feed to get article URLs and metadata."""
    try:
        r = requests.get(rss_url, headers=HEADERS, timeout=15)
        r.raise_for_status()
        root = ET.fromstring(r.content)
        articles = []
        for item in root.findall(".//item"):
            url   = item.findtext("link", "").strip()
            title = item.findtext("title", "").strip()
            date  = item.findtext("pubDate", "").strip()
            desc  = item.findtext("description", "").strip()
            if url and title:
                articles.append({"url": url, "title": title, "date": date, "desc": desc})
        return articles
    except Exception as e:
        print(f"  [WARN] RSS failed {rss_url}: {e}")
        return []

def scrape_bbc_article(url, meta):
    """Scrape full BBC Bangla article text."""
    try:
        r = requests.get(url, headers=HEADERS, timeout=15)
        r.raise_for_status()
        soup = BeautifulSoup(r.text, "lxml")

        # BBC article body selectors
        content = ""
        for sel in ["article[data-component='text-block']", "div[data-component='text-block']",
                    "div.ssrcss-11r1m41-RichTextComponentWrapper", "div.story-body__inner", "article"]:
            blocks = soup.select(sel)
            if blocks:
                content = " ".join(b.get_text(strip=True) for b in blocks if len(b.get_text(strip=True)) > 20)
                break

        # Fall back to all <p> tags in main content
        if len(content) < 100:
            main = soup.select_one("main, article, div[role='main']")
            if main:
                content = " ".join(p.get_text(strip=True) for p in main.find_all("p") if len(p.get_text(strip=True)) > 20)

        # Use RSS description as minimum fallback
        if len(content) < 50:
            content = meta.get("desc", "")

        if not content or len(content) < 50:
            return None

        return {
            "source": SOURCE,
            "url": url,
            "title": meta["title"],
            "content": content,
            "language": "bn",
            "published_date": meta["date"],
            "reliability_score": RELIABILITY[SOURCE],
            "scraped_at": datetime.utcnow().isoformat(),
        }
    except Exception as e:
        print(f"  [WARN] Failed to scrape {url}: {e}")
        return None

# Run
print("Fetching BBC Bangla RSS feeds...")
all_items = []
for rss in RSS_FEEDS:
    items = get_bbc_links_from_rss(rss)
    print(f"  {rss} → {len(items)} items")
    all_items.extend(items)

print(f"\nTotal articles found: {len(all_items)}")
print("Scraping full article text...")

saved = 0
failed = 0
for i, item in enumerate(all_items[:50]):
    article = scrape_bbc_article(item["url"], item)
    if article:
        if save_article(article, SOURCE):
            saved += 1
            print(f"  [{i+1}] SAVED: {article['title'][:60]}")
        else:
            print(f"  [{i+1}] SKIP (duplicate)")
    else:
        failed += 1
        print(f"  [{i+1}] FAIL: {item['url']}")
    time.sleep(1.5)

print(f"\nBBC Bangla — Saved: {saved} | Failed: {failed}")
print(f"Total in corpus: {count_saved()} articles")

Fetching BBC Bangla RSS feeds...
  https://feeds.bbci.co.uk/bengali/rss.xml → 14 items

Total articles found: 14
Scraping full article text...
  [1] SAVED: এআই ক্যামেরায় মামলার কথা জানিয়ে ফোনে বার্তা, কী করবেন?
  [2] SAVED: একই পশু দিয়ে আকিকা ও কোরবানি দেওয়া যাবে?
  [3] SAVED: বলিউডে নিষেধাজ্ঞার মুখে 'ধুরন্ধর' খ্যাত রণবীর সিং, কেন বিতর্
  [4] SAVED: ব্রাজিলের 'হেক্সা স্বপ্ন' কি এবার পূরণ হবে যুক্তরাষ্ট্রে?
  [5] SAVED: বাংলাদেশ থেকে ভারতে আসা মতুয়ারা  কেন নাগরিকত্ব পাওয়া নিয়ে নত
  [6] SAVED: ইরানের দক্ষিণে 'আত্মরক্ষামূলক' হামলা চালানোর দাবি যুক্তরাষ্ট
  [7] SAVED: পত্রিকা: 'খলিলুরের নজিরবিহীন ছুটির পরিকল্পনা নিয়ে বিস্ময়'
  [8] SAVED: প্রাচীন যেসব হজপথ দিয়ে দূর-দূরান্তের হাজিরা মক্কায় এসে সমবেত
  [9] SAVED: ঈদুল আযহার দিন বাংলাদেশের আবহাওয়া কেমন থাকবে?
  [10] SAVED: হজের সময় পালন করা ৮টি রীতির পেছনে যে ইতিহাস
  [11] SAVED: মুসলিমদের কাছে জমজম কূপের পানি কেন এতো গুরুত্বপূর্ণ ?
  [12] SAVED: ইসলাম প্রচারের পর কীভাবে কোরবানি দেয়া চালু হয়েছিল
  [13] SAVED: কোন পশু কোরবানি দেওয়া যাবে, আর  ক

## Source 3 — Prothom Alo
**URL:** https://www.prothomalo.com  
**Why:** Largest Bangla newspaper in Bangladesh — widest topic coverage.  
**Reliability:** 0.80  
**Strategy:** Try RSS feed first. Fall back to scraping category pages (bangladesh, national, world).

In [7]:
## Source 3 — Prothom Alo (via Sitemap XML)
# Prothom Alo is JS-rendered (Next.js SPA) — category page scraping doesn't work.
# Strategy: Use their sitemap to get direct article URLs, then scrape each article.

SOURCE = "prothom_alo"
BASE_URL = "https://www.prothomalo.com"

SITEMAP_URLS = [
    "https://www.prothomalo.com/sitemap.xml",
    "https://www.prothomalo.com/news-sitemap.xml",
]

def get_prothomalo_links_from_sitemap(sitemap_url):
    """Extract article URLs from Prothom Alo sitemap."""
    try:
        r = requests.get(sitemap_url, headers=HEADERS, timeout=15)
        r.raise_for_status()
        root = ET.fromstring(r.content)
        ns = {"sm": "http://www.sitemaps.org/schemas/sitemap/0.9"}
        urls = []
        for loc in root.findall(".//sm:loc", ns):
            u = loc.text.strip()
            # Only pick article URLs (have at least 3 path parts after domain)
            path = u.replace(BASE_URL, "")
            parts = [p for p in path.split("/") if p]
            if len(parts) >= 2 and not any(x in u for x in ["tag", "author", "category", "sitemap"]):
                urls.append(u)
        return urls
    except Exception as e:
        print(f"  [WARN] Sitemap failed {sitemap_url}: {e}")
        return []

def scrape_prothomalo_article(url):
    """Scrape a Prothom Alo article."""
    try:
        r = requests.get(url, headers=HEADERS, timeout=15)
        r.raise_for_status()
        soup = BeautifulSoup(r.text, "lxml")

        title = ""
        for sel in ["h1", "h1.title", "[class*='headline']", "[class*='title']"]:
            t = soup.select_one(sel)
            if t and len(t.get_text(strip=True)) > 5:
                title = t.get_text(strip=True)
                break

        content = ""
        # Try to find article body
        for sel in ["[class*='story-element-text']", "[class*='article-body']",
                    "[class*='content']", "article", "main"]:
            body = soup.select_one(sel)
            if body:
                paras = [p.get_text(strip=True) for p in body.find_all("p") if len(p.get_text(strip=True)) > 20]
                if paras:
                    content = " ".join(paras)
                    break

        date = ""
        d = soup.select_one("time[datetime], [class*='date'], [class*='time']")
        if d:
            date = d.get("datetime", d.get_text(strip=True))

        if not title or len(content) < 80:
            return None

        return {
            "source": SOURCE,
            "url": url,
            "title": title,
            "content": content[:3000],
            "language": "bn",
            "published_date": date,
            "reliability_score": RELIABILITY[SOURCE],
            "scraped_at": datetime.utcnow().isoformat(),
        }
    except Exception as e:
        print(f"  [WARN] {url}: {e}")
        return None

# Run
print("Fetching Prothom Alo sitemaps...")
all_links = []
for sm in SITEMAP_URLS:
    links = get_prothomalo_links_from_sitemap(sm)
    print(f"  {sm} → {len(links)} URLs")
    all_links.extend(links)

all_links = list(set(all_links))
print(f"Total unique article URLs: {len(all_links)}")

if not all_links:
    print("⚠️  Sitemap returned no links — Prothom Alo may block bots or sitemap URL changed.")
    print("Skipping Prothom Alo. Corpus is still valid with other sources.")
else:
    print(f"Scraping up to 40 articles...")
    saved = 0
    failed = 0
    for i, url in enumerate(all_links[:40]):
        article = scrape_prothomalo_article(url)
        if article:
            if save_article(article, SOURCE):
                saved += 1
                print(f"  [{i+1}] SAVED: {article['title'][:60]}")
            else:
                print(f"  [{i+1}] SKIP (dup)")
        else:
            failed += 1
        time.sleep(2)

    print(f"\nProthom Alo — Saved: {saved} | Failed: {failed}")
    print(f"Total in corpus: {count_saved()} articles")

Fetching Prothom Alo sitemaps...
  https://www.prothomalo.com/sitemap.xml → 0 URLs
  [WARN] Sitemap failed https://www.prothomalo.com/news-sitemap.xml: 404 Client Error: Not Found for url: https://www.prothomalo.com/news-sitemap.xml
  https://www.prothomalo.com/news-sitemap.xml → 0 URLs
Total unique article URLs: 0
⚠️  Sitemap returned no links — Prothom Alo may block bots or sitemap URL changed.
Skipping Prothom Alo. Corpus is still valid with other sources.


## Source 4 — The Daily Star BD
**URL:** https://www.thedailystar.net  
**Why:** Top English-language newspaper in Bangladesh. Covers BD events in English — good for English claims.  
**Reliability:** 0.80  
**Strategy:** RSS feeds by category (bangladesh, health, world) — very reliable, no blocking.

In [8]:
## Source 4 — The Daily Star BD (more RSS feeds)

SOURCE = "daily_star"
BASE_URL = "https://www.thedailystar.net"
RSS_FEEDS = [
    "https://www.thedailystar.net/rss.xml",
    "https://www.thedailystar.net/news/bangladesh/rss.xml",
    "https://www.thedailystar.net/opinion/rss.xml",
    "https://www.thedailystar.net/news/rss.xml",
    "https://www.thedailystar.net/health/rss.xml",
    "https://www.thedailystar.net/environment/rss.xml",
    "https://www.thedailystar.net/tech-startup/rss.xml",
]

def get_dailystar_links_from_rss(rss_url):
    try:
        r = requests.get(rss_url, headers=HEADERS, timeout=15)
        r.raise_for_status()
        root = ET.fromstring(r.content)
        articles = []
        for item in root.findall(".//item"):
            url   = item.findtext("link", "").strip()
            title = item.findtext("title", "").strip()
            date  = item.findtext("pubDate", "").strip()
            desc  = item.findtext("description", "").strip()
            desc_text = BeautifulSoup(desc, "lxml").get_text(strip=True)
            if url and title:
                articles.append({"url": url, "title": title, "date": date, "desc": desc_text})
        return articles
    except Exception as e:
        print(f"  [WARN] RSS {rss_url}: {e}")
        return []

def scrape_dailystar_article(url, meta):
    try:
        r = requests.get(url, headers=HEADERS, timeout=15)
        r.raise_for_status()
        soup = BeautifulSoup(r.text, "lxml")
        content = ""
        for sel in ["div.pb-20", "div[class*='article-body']", "div[class*='story-element']",
                    "div.field-items", "article", "main"]:
            body = soup.select_one(sel)
            if body:
                paras = [p.get_text(strip=True) for p in body.find_all("p") if len(p.get_text(strip=True)) > 20]
                if paras:
                    content = " ".join(paras)
                    break
        if len(content) < 50:
            content = meta.get("desc", "")
        if not content or len(content) < 50:
            return None
        return {
            "source": SOURCE,
            "url": url,
            "title": meta["title"],
            "content": content[:3000],
            "language": "en",
            "published_date": meta["date"],
            "reliability_score": RELIABILITY[SOURCE],
            "scraped_at": datetime.utcnow().isoformat(),
        }
    except Exception as e:
        print(f"  [WARN] {url}: {e}")
        return None

# Run
print("Fetching Daily Star RSS feeds...")
all_items = []
for rss in RSS_FEEDS:
    items = get_dailystar_links_from_rss(rss)
    print(f"  {rss} → {len(items)} items")
    all_items.extend(items)

seen = set()
unique_items = []
for item in all_items:
    if item["url"] not in seen:
        seen.add(item["url"])
        unique_items.append(item)

print(f"\nTotal unique: {len(unique_items)}")
saved = 0
failed = 0
for i, item in enumerate(unique_items[:50]):
    article = scrape_dailystar_article(item["url"], item)
    if article:
        if save_article(article, SOURCE):
            saved += 1
            print(f"  [{i+1}] SAVED: {article['title'][:60]}")
        else:
            print(f"  [{i+1}] SKIP (dup)")
    else:
        failed += 1
        print(f"  [{i+1}] FAIL: {item['title'][:50]}")
    time.sleep(1.5)

print(f"\nDaily Star — Saved: {saved} | Failed: {failed}")
print(f"Total in corpus: {count_saved()} articles")

Fetching Daily Star RSS feeds...
  https://www.thedailystar.net/rss.xml → 7 items
  https://www.thedailystar.net/news/bangladesh/rss.xml → 0 items
  https://www.thedailystar.net/opinion/rss.xml → 0 items
  https://www.thedailystar.net/news/rss.xml → 0 items
  https://www.thedailystar.net/health/rss.xml → 0 items
  https://www.thedailystar.net/environment/rss.xml → 0 items
  https://www.thedailystar.net/tech-startup/rss.xml → 0 items

Total unique: 7
  [1] SKIP (dup)
  [2] SKIP (dup)
  [3] SKIP (dup)
  [4] SKIP (dup)
  [5] SKIP (dup)
  [6] SKIP (dup)
  [7] SKIP (dup)

Daily Star — Saved: 0 | Failed: 0
Total in corpus: 69 articles


In [10]:
## Source 5 — Factwatch BD + Google News RSS (backup sources)

# ── Factwatch BD ──────────────────────────────────────────────────────────────
SOURCE_FACT = "factwatch"
RELIABILITY["factwatch"] = 0.92  # dedicated BD fact-checker

def crawl_factwatch(max_articles=30):
    base = "https://www.factwatch.org"
    saved = 0
    try:
        r = requests.get(f"{base}/fact-check/", headers=HEADERS, timeout=15)
        r.raise_for_status()
        soup = BeautifulSoup(r.text, "lxml")
        links = []
        for a in soup.find_all("a", href=True):
            href = a["href"]
            if "/fact-check/" in href and href != f"{base}/fact-check/":
                full = href if href.startswith("http") else base + href
                links.append(full)
        links = list(set(links))[:max_articles]
        print(f"  factwatch.org → {len(links)} article links")
        for i, url in enumerate(links):
            try:
                ar = requests.get(url, headers=HEADERS, timeout=15)
                ar.raise_for_status()
                s = BeautifulSoup(ar.text, "lxml")
                title = (s.select_one("h1") or s.select_one("h2") or type("", (), {"get_text": lambda *a,**k: ""})()).get_text(strip=True)
                paras = [p.get_text(strip=True) for p in s.find_all("p") if len(p.get_text(strip=True)) > 30]
                content = " ".join(paras[:20])
                if title and len(content) > 80:
                    article = {
                        "source": SOURCE_FACT, "url": url, "title": title,
                        "content": content[:3000], "language": "en",
                        "published_date": "", "reliability_score": RELIABILITY[SOURCE_FACT],
                        "scraped_at": datetime.utcnow().isoformat(),
                    }
                    if save_article(article, SOURCE_FACT):
                        saved += 1
                        print(f"  [{i+1}] SAVED: {title[:60]}")
                time.sleep(1.5)
            except Exception as e:
                print(f"  [{i+1}] FAIL: {e}")
    except Exception as e:
        print(f"  [WARN] factwatch.org: {e}")
    return saved

# ── Google News RSS (Bangladesh) ───────────────────────────────────────────────
SOURCE_GN = "google_news_bd"
RELIABILITY["google_news_bd"] = 0.75

GN_RSS_URLS = [
    "https://news.google.com/rss/search?q=Bangladesh+news+fact+check&hl=en-BD&gl=BD&ceid=BD:en",
    "https://news.google.com/rss/search?q=Bangladesh+misinformation&hl=en&gl=BD&ceid=BD:en",
]

def crawl_google_news(max_articles=30):
    saved = 0
    all_items = []
    for rss in GN_RSS_URLS:
        try:
            r = requests.get(rss, headers=HEADERS, timeout=15)
            r.raise_for_status()
            root = ET.fromstring(r.content)
            for item in root.findall(".//item"):
                url   = item.findtext("link", "").strip()
                title = item.findtext("title", "").strip()
                date  = item.findtext("pubDate", "").strip()
                desc  = BeautifulSoup(item.findtext("description", ""), "lxml").get_text(strip=True)
                if url and title:
                    all_items.append({"url": url, "title": title, "date": date, "desc": desc})
            print(f"  {rss[:60]}... → {len(all_items)} items")
        except Exception as e:
            print(f"  [WARN] Google News: {e}")

    seen = set()
    for item in all_items:
        if item["url"] in seen:
            continue
        seen.add(item["url"])
        try:
            r = requests.get(item["url"], headers=HEADERS, timeout=15)
            r.raise_for_status()
            s = BeautifulSoup(r.text, "lxml")
            paras = [p.get_text(strip=True) for p in s.find_all("p") if len(p.get_text(strip=True)) > 30]
            content = " ".join(paras[:20]) or item["desc"]
            if len(content) > 80:
                article = {
                    "source": SOURCE_GN, "url": item["url"], "title": item["title"],
                    "content": content[:3000], "language": "en",
                    "published_date": item["date"], "reliability_score": RELIABILITY[SOURCE_GN],
                    "scraped_at": datetime.utcnow().isoformat(),
                }
                if save_article(article, SOURCE_GN):
                    saved += 1
                    print(f"  SAVED: {item['title'][:60]}")
            time.sleep(1.5)
            if saved >= max_articles:
                break
        except Exception as e:
            print(f"  FAIL: {e}")
    return saved

print("=" * 50)
print("Crawling Factwatch BD...")
s1 = crawl_factwatch(30)
print(f"\nFactwatch saved: {s1}")

print("\nCrawling Google News BD...")
s2 = crawl_google_news(30)
print(f"\nGoogle News saved: {s2}")

print(f"\nTotal in corpus: {count_saved()} articles")

Crawling Factwatch BD...
  [WARN] factwatch.org: HTTPSConnectionPool(host='www.factwatch.org', port=443): Max retries exceeded with url: /fact-check/ (Caused by NameResolutionError("HTTPSConnection(host='www.factwatch.org', port=443): Failed to resolve 'www.factwatch.org' ([Errno 11001] getaddrinfo failed)"))

Factwatch saved: 0

Crawling Google News BD...
  https://news.google.com/rss/search?q=Bangladesh+news+fact+ch... → 100 items
  https://news.google.com/rss/search?q=Bangladesh+misinformati... → 200 items
  SAVED: Bangladesh’s FDI reality check: Reform must move from promis
  SAVED: Does This Video Show a Demolition Drive in West Bengal? No, 
  SAVED: Bangladesh protest video falsely shared as Indian election v
  SAVED: Fact Check: Old Video From Bangladesh Viral As Madhya Prades
  SAVED: Rumor Scanner identifies viral fake photocard targeting poli
  SAVED: 29 false info about Tarique Rahman identified on Facebook in
  SAVED: False internet shutdown claims spread before Bangladesh'

## Final Summary
Run this cell after all sources are crawled to see the total corpus size and breakdown per source.

In [12]:
## Source 6 — More Rumor Scanner BD pages (pages 6–15)
# We only scraped pages 1-5 before. More pages = more fact-check articles.

EXTRA_RS_PAGES = [
    f"https://www.rumorscanner.com/category/fact-check/page/{n}/" for n in range(6, 16)
]

print("Collecting more Rumor Scanner BD links (pages 6-15)...")
extra_links = []
for page in EXTRA_RS_PAGES:
    links = get_rumorscanner_links(page)
    print(f"  {page} → {len(links)} links")
    extra_links.extend(links)
    time.sleep(1)

extra_links = list(set(extra_links))
print(f"\nNew unique links: {len(extra_links)}")

saved = 0
failed = 0
for i, url in enumerate(extra_links):
    article = scrape_rumorscanner_article(url)
    if article:
        if save_article(article, "rumorscanner"):
            saved += 1
            print(f"  [{i+1}] SAVED: {article['title'][:60]}")
        else:
            print(f"  [{i+1}] SKIP (dup)")
    else:
        failed += 1
    time.sleep(1.5)

print(f"\nRumor Scanner extra pages — Saved: {saved} | Failed: {failed}")
print(f"Total in corpus: {count_saved()} articles")

  https://www.rumorscanner.com/category/fact-check/page/6/ → 12 links
  https://www.rumorscanner.com/category/fact-check/page/7/ → 12 links
  https://www.rumorscanner.com/category/fact-check/page/8/ → 12 links
  https://www.rumorscanner.com/category/fact-check/page/9/ → 12 links
  https://www.rumorscanner.com/category/fact-check/page/10/ → 12 links
  https://www.rumorscanner.com/category/fact-check/page/11/ → 12 links
  https://www.rumorscanner.com/category/fact-check/page/12/ → 12 links
  https://www.rumorscanner.com/category/fact-check/page/13/ → 12 links
  https://www.rumorscanner.com/category/fact-check/page/14/ → 12 links
  https://www.rumorscanner.com/category/fact-check/page/15/ → 12 links

New unique links: 93
  [1] SAVED: বিএনপি কর্তৃক মহিলা যুবলীগের নেত্রীকে মারধরের ঘটনা দাবিতে চু
  [2] SAVED: সেতু-ফ্লাইওভারে শেখ হাসিনার গ্রাফিতি দাবিতে এআই ছবি প্রচার
  [3] SAVED: কুমিল্লার ভিন্ন ঘটনাকে আসামে হিন্দুর ঘরে বাংলাদেশি মুসলমানের
  [4] SAVED: ইন্দোনেশিয়ায় বন্যায় বাঘকে রক্ষায় পি

In [13]:
## Source 7 — Dhaka Tribune (English BD newspaper with RSS)

SOURCE_DT = "dhaka_tribune"
RELIABILITY["dhaka_tribune"] = 0.80

DT_RSS_FEEDS = [
    "https://www.dhakatribune.com/feed",
    "https://www.dhakatribune.com/bangladesh/feed",
    "https://www.dhakatribune.com/health/feed",
    "https://www.dhakatribune.com/world/feed",
]

def scrape_dhaka_tribune(rss_list, max_articles=40):
    all_items = []
    for rss in rss_list:
        try:
            r = requests.get(rss, headers=HEADERS, timeout=15)
            r.raise_for_status()
            root = ET.fromstring(r.content)
            for item in root.findall(".//item"):
                url   = item.findtext("link", "").strip()
                title = item.findtext("title", "").strip()
                date  = item.findtext("pubDate", "").strip()
                desc  = BeautifulSoup(item.findtext("description", ""), "lxml").get_text(strip=True)
                if url and title:
                    all_items.append({"url": url, "title": title, "date": date, "desc": desc})
            print(f"  {rss} → {len(all_items)} items so far")
        except Exception as e:
            print(f"  [WARN] {rss}: {e}")

    seen = set()
    unique = [x for x in all_items if not (x["url"] in seen or seen.add(x["url"]))]
    print(f"  Total unique: {len(unique)}")

    saved = 0
    for i, item in enumerate(unique[:max_articles]):
        try:
            r = requests.get(item["url"], headers=HEADERS, timeout=15)
            r.raise_for_status()
            s = BeautifulSoup(r.text, "lxml")
            content = ""
            for sel in ["div.entry-content", "div[class*='article']", "div[class*='body']", "article"]:
                body = s.select_one(sel)
                if body:
                    paras = [p.get_text(strip=True) for p in body.find_all("p") if len(p.get_text(strip=True)) > 20]
                    if paras:
                        content = " ".join(paras)
                        break
            if len(content) < 50:
                content = item["desc"]
            if len(content) < 50:
                continue
            article = {
                "source": SOURCE_DT, "url": item["url"], "title": item["title"],
                "content": content[:3000], "language": "en",
                "published_date": item["date"], "reliability_score": RELIABILITY[SOURCE_DT],
                "scraped_at": datetime.utcnow().isoformat(),
            }
            if save_article(article, SOURCE_DT):
                saved += 1
                print(f"  [{i+1}] SAVED: {item['title'][:60]}")
            else:
                print(f"  [{i+1}] SKIP (dup)")
        except Exception as e:
            print(f"  [{i+1}] FAIL: {e}")
        time.sleep(1.5)
    return saved

print("Crawling Dhaka Tribune...")
s = scrape_dhaka_tribune(DT_RSS_FEEDS, 40)
print(f"\nDhaka Tribune — Saved: {s}")
print(f"Total in corpus: {count_saved()} articles")

Crawling Dhaka Tribune...
  [WARN] https://www.dhakatribune.com/feed: 404 Client Error: Not Found for url: https://www.dhakatribune.com/feed
  [WARN] https://www.dhakatribune.com/bangladesh/feed: 404 Client Error: Not Found for url: https://www.dhakatribune.com/bangladesh/feed
  [WARN] https://www.dhakatribune.com/health/feed: 404 Client Error: Not Found for url: https://www.dhakatribune.com/health/feed
  [WARN] https://www.dhakatribune.com/world/feed: 404 Client Error: Not Found for url: https://www.dhakatribune.com/world/feed
  Total unique: 0

Dhaka Tribune — Saved: 0
Total in corpus: 189 articles


In [14]:
## Final Summary — Corpus Overview

from collections import Counter

all_files = list(RAW_DIR.glob("*.json"))
source_counts = Counter()
total_chars = 0

for f in all_files:
    with open(f, encoding="utf-8") as fp:
        data = json.load(fp)
    source_counts[data.get("source", "unknown")] += 1
    total_chars += len(data.get("content", ""))

print("=" * 50)
print("CORPUS SUMMARY")
print("=" * 50)
print(f"Total articles : {len(all_files)}")
print(f"Total text     : ~{total_chars // 1000}K characters")
print()
print("By source:")
for source, count in source_counts.most_common():
    print(f"  {source:20s} : {count} articles")

print()
if len(all_files) >= 50:
    print("✅ Corpus ready for chunking (Todo #4)")
elif len(all_files) >= 20:
    print("⚠️  Small corpus — enough for demo, consider running more pages")
else:
    print("❌ Too few articles — re-run crawlers or check for errors above")

CORPUS SUMMARY
Total articles : 189
Total text     : ~400K characters

By source:
  daily_star           : 97 articles
  rumorscanner         : 48 articles
  google_news_bd       : 30 articles
  bbc_bangla           : 14 articles

✅ Corpus ready for chunking (Todo #4)
